In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Conf To Disable AQE

In [0]:
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")

# spark.conf.set("spark.sql.adaptive.enabled", "false")  # AQE Disable

# 🚀 Spark Jobs, Stages & Tasks (Interview-Level Deep Dive)

---

# 1️⃣ What is a Job in Spark?

## 📌 Definition

> A Job is triggered when an **action** is called on an RDD or DataFrame.

---

## 🔹 Examples of Actions

```python
df.show()
df.collect()
df.count()
df.write.parquet("/path")
```

Each action → **New Job**

---

## 🔹 Example

```python
df = spark.read.csv("file.csv")

df_filtered = df.filter("age > 25")

df_filtered.show()   # Job 1
df_filtered.count()  # Job 2
```

👉 Two actions → **Two separate jobs**

---

## 🔹 Key Points

- One action = One job  
- Job is the **top-level execution unit**  
- Visible in **Spark UI → Jobs tab**

---

# 2️⃣ What is a Stage?

## 📌 Definition

> A Stage is a group of tasks that can be executed **without shuffle**.

---

## 🔹 How Stages Are Created?

- Spark divides a job into stages
- Each **wide transformation (shuffle)** creates a new stage

---

## 🔹 Example

```python
rdd = spark.sparkContext.parallelize([1,2,3,4])

rdd2 = rdd.map(lambda x: x * 2)
rdd3 = rdd2.filter(lambda x: x > 2)
rdd4 = rdd3.reduceByKey(lambda x,y: x+y)

rdd4.collect()
```

---

## 🔹 Stage Breakdown

```
Stage 1: map → filter   (Narrow transformations)
Stage 2: reduceByKey    (Wide transformation → shuffle)
```

---

## 🔹 Key Points

- Narrow transformations → Same stage  
- Wide transformations → New stage  
- Shuffle = Stage boundary  

---

# 3️⃣ What is a Task?

## 📌 Definition

> A Task is the smallest unit of execution in Spark.

---

## 🔹 Key Rule

👉 One partition = One task  

---

## 🔹 Example

```python
rdd = spark.sparkContext.parallelize(range(1, 101), 4)

rdd.map(lambda x: x * 2).collect()
```

- Total partitions = 4  
- Total tasks = 4  

Each task processes one partition.

---

## 🔹 Where Tasks Run?

- Inside **Executors**
- In parallel across cluster

---

# 4️⃣ Complete Flow: Job → Stage → Task

---

## 🔹 Execution Flow

```
Action Triggered
      ↓
Job Created
      ↓
DAG Scheduler splits into stages
      ↓
Task Scheduler creates tasks
      ↓
Tasks assigned to Executors
      ↓
Execution happens
```

---

## 🔹 Visual Representation

```
Job
 ├── Stage 1 (No Shuffle)
 │     ├── Task 1
 │     ├── Task 2
 │     └── Task 3
 │
 └── Stage 2 (Shuffle)
       ├── Task 1
       ├── Task 2
       └── Task 3
```

---

# 5️⃣ Hands-On Example (Very Important)

```python
df = spark.range(0, 100)

df2 = df.filter("id > 10")      # Narrow
df3 = df2.groupBy("id").count() # Wide

df3.show()
```

---

## 🔹 What Happens Internally?

### Step 1: Action Triggered

```
show() → Job created
```

---

### Step 2: DAG Created

```
range → filter → groupBy → count
```

---

### Step 3: Stage Division

```
Stage 1: range + filter
Stage 2: groupBy (shuffle)
```

---

### Step 4: Tasks Created

If 4 partitions:

```
Stage 1 → 4 tasks
Stage 2 → 4 tasks
```

---

# 6️⃣ How to See This in Spark UI

---

## 🔹 Steps

1. Go to Databricks notebook  
2. Run any action (`show()`, `display()`)  
3. Click → "View Spark UI"  

---

## 🔹 What You Will See

### Jobs Tab

- List of jobs triggered  
- Execution time  

---

### Stages Tab

- Stage breakdown  
- Shuffle read/write  
- Task count  

---

### Tasks View

- Task duration  
- Input size  
- Skew detection  

---

# 7️⃣ Important Interview Concepts

---

## 🔹 Job vs Stage vs Task

| Level | Description |
|--------|-------------|
| Job | Triggered by action |
| Stage | Split by shuffle |
| Task | Runs on partition |

---

## 🔹 Key Relationships

- One Job → Multiple Stages  
- One Stage → Multiple Tasks  
- One Task → One Partition  

---

# 8️⃣ Advanced Concepts

---

## 🔹 Stage Types

- **Shuffle Map Stage** → Produces shuffle data  
- **Result Stage** → Final stage returning output  

---

## 🔹 Example

```
Stage 1 → Shuffle Map Stage
Stage 2 → Result Stage
```

---

## 🔹 DAG Scheduler vs Task Scheduler

### DAG Scheduler

- Splits job into stages  
- Handles shuffle dependencies  

---

### Task Scheduler

- Assigns tasks to executors  
- Handles execution  

---

# 9️⃣ Real-World Insight

---

## 🔹 Performance Impact

- More partitions → More tasks → Better parallelism  
- Too many tasks → Overhead  
- Shuffle → Expensive  

---

## 🔹 Optimization Tips

- Reduce shuffle where possible  
- Use broadcast joins  
- Tune partitions  
- Monitor Spark UI  

---

# 🎯 Interview-Level Summary

- Action → Creates Job  
- Job → Split into Stages  
- Stage → Split into Tasks  
- Task → Executes on partition  
- Shuffle → Creates new stage  
- Narrow → Same stage  
- Wide → New stage  

---

# 🚀 Final Understanding

```
Action
  ↓
Job
  ↓
Stages (Based on Shuffle)
  ↓
Tasks (Based on Partitions)
  ↓
Executors execute tasks
```

---

# 🔥 Golden Rule (Must Remember)

👉 One Partition = One Task  
👉 One Action = One Job  
👉 One Shuffle = New Stage  

In [0]:
dbutils.fs.ls("dbfs:/FileStore/tables/Arijit/Test/")

In [0]:
df = spark.read.format("csv").option("header", True)\
                            .option("inferSchema", True)\
                            .load("dbfs:/FileStore/tables/Arijit/Test/MegaMart.csv")

In [0]:
df.display()

In [0]:
# Transformations

df = df.filter(col('product_name') == 'Sneakers')

In [0]:
df = df.select('order_id', 'product_name')

In [0]:
df = df.groupBy('product_name').agg(count(col('order_id')))

In [0]:
display(df)

# 🚀 Spark Joins Deep Dive (Interview-Level)

---

# 1️⃣ What is a Join in Spark?

## 📌 Definition

> A join combines two datasets based on a common key.

---

## 🔹 Example

```python
df1.join(df2, "id")
```

---

# 2️⃣ Types of Joins in Spark

---

## 🔹 1. Inner Join

Returns matching records from both tables.

```python
df1.join(df2, "id", "inner")
```

---

## 🔹 2. Left Join

All records from left + matching from right.

```python
df1.join(df2, "id", "left")
```

---

## 🔹 3. Right Join

All records from right + matching from left.

```python
df1.join(df2, "id", "right")
```

---

## 🔹 4. Full Outer Join

All records from both sides.

```python
df1.join(df2, "id", "outer")
```

---

## 🔹 5. Left Semi Join

Only matching rows from left (no columns from right).

```python
df1.join(df2, "id", "left_semi")
```

---

## 🔹 6. Left Anti Join

Rows from left that DO NOT match.

```python
df1.join(df2, "id", "left_anti")
```

---

# 3️⃣ How Join Works Internally

---

## 🔥 Step-by-Step Execution

```
1. Read both datasets
2. Partition data based on join key
3. Shuffle data across executors
4. Bring same keys together
5. Perform join
```

---

## 🔹 Why Shuffle Happens?

Because:

- Same keys must be in same partition
- Data initially distributed randomly

---

## 🔹 Example

```
df1:
Partition 1 → id: 1,2
Partition 2 → id: 3,4

df2:
Partition 1 → id: 3,4
Partition 2 → id: 1,2
```

Before join → keys are scattered

After shuffle:

```
Partition 1 → id: 1,2
Partition 2 → id: 3,4
```

Now join is possible.

---

# 4️⃣ Hash Partitioning (Very Important 🔥)

---

## 📌 What is Hash Partitioning?

Spark distributes data using:

```
partition = hash(key) % num_partitions
```

---

## 🔹 Example

```python
hash("A") % 4 = 1
hash("B") % 4 = 3
```

So:

- All "A" → Partition 1
- All "B" → Partition 3

---

## 🔹 Why Important?

- Ensures same keys go to same partition
- Enables join & aggregation
- Used during shuffle

---

# 5️⃣ Join Strategies (Most Important 🔥🔥🔥)

Spark uses different strategies based on data size.

---

# 🔹 1. Broadcast Hash Join (Best for small table)

---

## 📌 When Used?

- One table is small (< 10MB default)

---

## 🔹 Example

```python
from pyspark.sql.functions import broadcast

df1.join(broadcast(df2), "id")
```

---

## 🔹 How It Works?

```
Small table → Sent to all executors
Large table → Stays distributed
Join happens locally
```

---

## 🔹 Advantages

- No shuffle for large table
- Very fast

---

# 🔹 2. Sort Merge Join (Default for large data)

---

## 📌 When Used?

- Both tables are large

---

## 🔹 Steps

```
1. Shuffle both datasets
2. Sort data by key
3. Merge join
```

---

## 🔹 Characteristics

- Requires shuffle
- Requires sorting
- More expensive

---

# 🔹 3. Shuffle Hash Join

---

## 📌 When Used?

- Smaller data compared to SMJ
- Enough memory available

---

## 🔹 Steps

```
1. Shuffle both datasets
2. Build hash table on smaller partition
3. Probe with larger dataset
```

---

# 🔹 4. Broadcast Nested Loop Join

---

## 📌 When Used?

- No join key (cross join)
- Non-equi joins

---

## 🔹 Very expensive (avoid)

---

# 6️⃣ How to Check Join Type

---

```python
df.join(df2, "id").explain(True)
```

Look for:

- BroadcastHashJoin
- SortMergeJoin
- ShuffledHashJoin

---

# 7️⃣ Shuffle in Join (Deep Concept)

---

## 🔹 Why Shuffle Happens?

Because:

- Data is not aligned by key
- Needs redistribution

---

## 🔹 Shuffle Cost

- Disk I/O
- Network transfer
- Serialization
- Memory usage

---

## 🔹 Spark Config

```python
spark.conf.get("spark.sql.shuffle.partitions")
```

Default = 200

---

# 8️⃣ Data Skew in Join (Very Important)

---

## 🔹 What is Skew?

Uneven key distribution.

Example:

```
Key "A" → 90% data
Key "B" → 10% data
```

---

## 🔹 Problem

- One partition overloaded
- One task slow
- Others idle

---

## 🔹 Solutions

- Broadcast join
- Salting technique
- Repartition
- Skew join optimization (Spark 3+)

---

# 9️⃣ Partitioning Optimization

---

## 🔹 Pre-Partition Data

```python
df1 = df1.repartition("id")
df2 = df2.repartition("id")
```

Helps reduce shuffle.

---

## 🔹 Bucketing (Advanced)

- Pre-partition data on disk
- Used in Hive tables

---

# 🔟 Hands-On Example

---

```python
df1 = spark.range(0, 1000000)
df2 = spark.range(0, 1000)

# Normal join (shuffle)
df1.join(df2, "id").explain()

# Broadcast join (optimized)
from pyspark.sql.functions import broadcast
df1.join(broadcast(df2), "id").explain()
```

---

# 🎯 Interview-Level Key Points

- Join is usually a **wide transformation**
- Shuffle is required to align keys
- Hash partitioning ensures correct grouping
- Broadcast join avoids shuffle
- Sort merge join is default for large data
- Skew is biggest performance issue
- Always check execution plan

---

# 🚀 Final Summary

```
Join
  ↓
Shuffle (if needed)
  ↓
Hash partitioning
  ↓
Join strategy selected
  ↓
Execution
```

---

# 🔥 Golden Rules

- Small table → Broadcast join  
- Large tables → Sort merge join  
- Avoid shuffle when possible  
- Watch skew carefully  
- Always check df.explain()

# 🚀 Advanced Spark Join Optimization (Deep Dive)

---

# 1️⃣ Broadcast Join vs Sort Merge Join (Real Understanding)

---

## 🔹 Broadcast Hash Join (BHJ)

### 📌 When Used?

- One dataset is small (default < 10MB)
- Can fit in memory

---

### 🔹 Example

```python
from pyspark.sql.functions import broadcast

df_large = spark.range(0, 1000000)
df_small = spark.range(0, 1000)

df_large.join(broadcast(df_small), "id").explain(True)
```

---

### 🔹 How It Works

```
Small table → Broadcast to all executors
Large table → Remains distributed
Join happens locally
```

---

### 🔹 Benefits

- No shuffle for large dataset
- Very fast
- Low network cost

---

## 🔹 Sort Merge Join (SMJ)

### 📌 When Used?

- Both datasets are large
- Default join strategy

---

### 🔹 Example

```python
df1 = spark.range(0, 1000000)
df2 = spark.range(0, 1000000)

df1.join(df2, "id").explain(True)
```

---

### 🔹 How It Works

```
1. Shuffle both datasets
2. Sort each partition
3. Merge join
```

---

### 🔹 Drawback

- Expensive (shuffle + sort)
- High disk + network I/O

---

## 🔥 Key Comparison

| Feature | Broadcast Join | Sort Merge Join |
|----------|----------------|------------------|
| Shuffle | ❌ No (big table) | ✅ Yes |
| Speed | Fast | Slower |
| Use Case | Small + Large | Large + Large |

---

# 2️⃣ Salting Technique (Handle Data Skew)

---

## 🔹 Problem: Data Skew

Example:

```
Key "A" → 90% data
Key "B" → 10% data
```

One partition overloaded → slow job.

---

## 🔹 Solution: Salting

Add random suffix to skewed keys.

---

## 🔹 Step-by-Step Example

### Step 1: Add Salt Column

```python
from pyspark.sql.functions import rand, floor

df1_salted = df1.withColumn("salt", floor(rand() * 5))
```

---

### Step 2: Expand Small Dataset

```python
from pyspark.sql.functions import explode, array

df2_salted = df2.withColumn("salt", explode(array([0,1,2,3,4])))
```

---

### Step 3: Join Using Salt + Key

```python
df_joined = df1_salted.join(df2_salted, ["id", "salt"])
```

---

## 🔹 What Happens?

- Skewed key "A" split into multiple partitions
- Load distributed evenly

---

# 3️⃣ Skew Join Handling (Spark 3+)

---

## 🔹 Automatic Skew Handling

Spark 3 introduced:

```
Adaptive Query Execution (AQE)
```

---

## 🔹 Enable It

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

## 🔹 What Spark Does

- Detects skewed partitions
- Splits large partitions
- Joins them separately

---

## 🔹 Benefit

- No manual salting required
- Automatic optimization

---

# 4️⃣ Bucketing vs Partitioning

---

# 🔹 Partitioning

Data physically divided by column.

```python
df.write.partitionBy("country").parquet("/path")
```

---

## 🔹 Use Case

- Query pruning
- Faster filtering

---

## 🔹 Bucketing

Data divided into fixed number of buckets using hash.

```python
df.write.bucketBy(8, "id").saveAsTable("bucketed_table")
```

---

## 🔹 Key Difference

| Feature | Partitioning | Bucketing |
|----------|--------------|-------------|
| Based on | Column value | Hash |
| File structure | Folder-based | Fixed buckets |
| Use case | Filtering | Joins |

---

## 🔹 Why Bucketing Helps Joins?

If both tables:

- Bucketed on same key
- Same number of buckets

👉 Shuffle can be avoided.

---

# 5️⃣ Real-World Join Optimization Case Study

---

## 🔹 Problem

- Large table: 1 Billion rows
- Small table: 1 Million rows
- Join taking too long

---

## 🔹 Initial Code

```python
df_large.join(df_small, "id")
```

---

## 🔹 Issues

- Sort Merge Join
- Full shuffle
- High execution time

---

## 🔹 Optimized Solution

```python
from pyspark.sql.functions import broadcast

df_large.join(broadcast(df_small), "id")
```

---

## 🔹 Result

- Broadcast Hash Join used
- Shuffle avoided
- Execution time reduced drastically

---

## 🔹 Additional Optimization

- Reduce shuffle partitions:

```python
spark.conf.set("spark.sql.shuffle.partitions", "50")
```

---

## 🔹 If Skew Exists

Apply:

- Salting
- AQE skew handling

---

# 🎯 Interview-Level Key Points

- Broadcast join → Best for small table
- Sort merge join → Default for large data
- Skew → Major performance issue
- Salting → Manual skew handling
- AQE → Automatic skew handling
- Bucketing → Avoid shuffle in joins
- Partitioning → Helps filtering

---

# 🚀 Final Summary

```
Join Optimization Strategy:

Small table → Broadcast
Large tables → Sort Merge
Skew → Salting / AQE
Pre-partitioned → Less shuffle
Bucketed tables → No shuffle (ideal case)
```

---

# 🔥 Golden Rules

- Always check df.explain()
- Avoid unnecessary shuffle
- Use broadcast wisely
- Watch skew carefully
- Tune shuffle partitions

# 🚀 Spark Join Hands On

## Disabling Auto Broadcast Join & AQE For Understanding Normal Sort Merge Join

In [0]:
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.enabled", False)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType
# schema = StructType([
#     StructField('order_id', LongType(), True),
#     StructField('product_name', StringType(), True),
#     StructField('order_date', StringType(), True),
#     StructField('order_customer_id', LongType(), True),
#     StructField('order_status', StringType(), True)
# ])

# Create First DataFrame
data1 = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eve")
]

df1 = spark.createDataFrame(data1, ["id", "name"])

# Create Second DataFrame
data2 = [
    (1, 50000),
    (2, 60000),
    (3, 70000),
    (6, 80000),
    (8, 90000)
]

df2 = spark.createDataFrame(data2, ["id", "salary"])



## Sort Merge Join

![image_1775982330032.png](./Images/image_1775982330032.png "image_1775982330032.png")

In [0]:
# df_join = df1.join(df2, df1['id'] == df2['id'], how="left")
df_join = df1.join(df2, df1.id == df2.id, how="left")

In [0]:
df_join.display()

## Broadcast Hash Join

![image_1775982296625.png](./Images/image_1775982296625.png "image_1775982296625.png")

In [0]:
from pyspark.sql.functions import broadcast

df_join_broad = df1.join(broadcast(df2), df1.id == df2.id, how="left")

In [0]:
df_join_broad.display()